In [46]:
from pyspark.sql import functions as f
from datetime import datetime

In [47]:
bi_carteira_cobranca = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_CARTEIRA_COBRANCA/')

In [48]:
bi_contrato = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_CONTRATO/')

In [49]:
bi_recebimento = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_RECEBIMENTOS/')

In [50]:
base_zeus = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BASE_ZEUS/')

In [51]:
titulos_vertia = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_TITULOS_VERTIA/')

In [52]:
bi_originais = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ORIGINAIS/')

In [53]:
bi_acordos_promessas = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ACORDOS_PROMESSAS/')

In [54]:
polos_unidades = spark.read.format("csv").option("delimiter", ";").option("header","true").load('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/POLOS_PARCEIROS_E_PROPRIOS/')

In [10]:
# JOIN >>

In [11]:
#4) cruzar com a tabela de ACORDOS_PROMESSA ( cod_parcelamento ) E TRAZER OS CAMPOS ABAIXO	
#				NOM_CARTEIRA
#				SIT_PARCELAMENTO
#				VL_PRINCIPAL
#				VL_DESC_PRINCIPAL
#				VL_MULTA
#				VL_DESC_MULTA
#				VL_JUROS_MORA
#				VL_DESC_JUROS_MORA
#				VL_JUROS_FINANC
#				VL_DESC_JUROS_FINANC
#				VL_CORRECAO_MONETARIA
#				VL_DESC_CORRECAO_MONETARIA
#				WHERE R.NR_NO_PARCELA = 1
def join_recebimento(bi_recebimento, bi_acordos_promessas):
    bi_acordos_promessas = bi_acordos_promessas.filter(f.col("NUM_QTDE_PARCELAS") == 1 )
    
    df_bi_recebimento = (bi_recebimento.join(bi_acordos_promessas, 
                           bi_recebimento['cod_parcelamento'] == bi_acordos_promessas['cod_parcelamento'], "left"))
    return df_bi_recebimento.drop(bi_acordos_promessas['cod_parcelamento']).drop(bi_acordos_promessas['COD_CONTRATO'])

In [12]:
df_bi_recebimento = join_recebimento(bi_recebimento, bi_acordos_promessas)

In [13]:
#	5) 	
#	update C
#		SET C.NOM_AGENCIA_EXTERNA = B.NOM_AGENCIA_EXTERNA
#	from  df_bi_recebimentos C WITH(NOLOCK)
#    LEFT join df_bi_carteira_cobranca  B  ON C.CD_ID_CONTRATO  = B.ID_CONTRATO;

def join_carteira_cobranca(df_bi_recebimento, bi_carteira_cobranca):
    df_bi_recebimento = (df_bi_recebimento.join(bi_carteira_cobranca, 
                           df_bi_recebimento['COD_CONTRATO'] == bi_carteira_cobranca['COD_CONTRATO'], "left"))
    return df_bi_recebimento.drop(bi_carteira_cobranca['COD_CONTRATO']).drop(bi_carteira_cobranca['TIP_MATRICULA'])

In [14]:
df_bi_recebimento = join_carteira_cobranca(df_bi_recebimento, bi_carteira_cobranca)

In [15]:
#6) CRIAR O CAMPO CDD_CPFCGC_PES e alimentar ele com a informação da tabela BI_CONTRATO

#7) Cruzar com a Zeus e trazer o    -- TIPO CURSO
#	UPDATE T 
#		SET	T.TIP_CURSO =  GA.TIP_CURSO
#	FROM  df_bi_recebimentos T WITH(NOLOCK)
#	INNER JOIN BASE_ZEUS GA with(nolock) on T.CPFCGC_PES = GA.COD_RA_ALUNO

def join_base_zeus(df_bi_recebimento, base_zeus):
    df_bi_recebimento =  (df_bi_recebimento.withColumn("COD_CPFCGC_PES", f.when(f.col("TIP_MATRICULA") == "PRESENCIAL",
                        f.concat(
                        f.lpad(f.col("COD_FILIAL"),3,'0'), f.lit(""), f.substring(f.col('COD_CONTRATO_ORIGEM'), 2, 15)))
                         .otherwise(f.concat(f.lit("EAD"),f.lit(""), f.col("COD_CONTRATO_ORIGEM")))))
    
    df_bi_recebimento = (df_bi_recebimento.join(base_zeus, 
                           df_bi_recebimento['COD_CPFCGC_PES'] == base_zeus['COD_RA_ZEUS'], "left"))
    return df_bi_recebimento.drop(base_zeus['COD_RA_ZEUS'])

In [16]:
df_bi_recebimento = join_base_zeus(df_bi_recebimento, base_zeus)

In [17]:
#8 ) -- MARCA ALUNOS VERTIA
#	FROM df_bi_recebimentos T with(nolock)
#	LEFT JOIN  TITULOS_VERTIA  V with(nolock) ON LTRIM(RTRIM(T.CPFCGC_PES)) = LTRIM(RTRIM(V.CPFCGC_PES))

def join_titulos_vertia(df_bi_recebimento, titulos_vertia):
    df_bi_recebimento = (df_bi_recebimento.join(titulos_vertia, 
                           f.ltrim(f.rtrim(df_bi_recebimento['COD_CPFCGC_PES'])) == f.ltrim(f.rtrim(titulos_vertia['COD_CPFCGC_PES'])), "left"))
    return df_bi_recebimento.drop(titulos_vertia['COD_CPFCGC_PES']).drop(titulos_vertia["COD_CONTRATO"])

In [18]:
df_bi_recebimento = join_titulos_vertia(df_bi_recebimento, titulos_vertia)

In [19]:
#9) UPDATE A
#	CRIAR CAMPO EM RECEBIMENTO: NATUREZA_TITULO = "MENSALIDADE"
#								NATUREZA_REMATRICULA = "S"

#9) INNER JOIN RAID.BI_ORIGINAIS O WITH(NOLOCK) 
#			ON		A.CD_ID_PARCELAMENTO	= O.ID_PARCELAMENTO	
#			AND		O.NOM_NATUREZA_TITULO		= 'MENSALIDADE'		 
#			AND		O.IND_NATUREZA_REMATRICULA	= 'S'			 
#			AND		O.ANO_COMPETENCIA		= 2022
#	INNER JOIN RAID.RAID.GC_ACCOUNT == bi_contrato GC WITH(NOLOCK) ON GC.ID_CONTRATO = A.CD_ID_CONTRATO 

def join_originais(df_bi_recebimento, bi_originais):    
    df_bi_recebimento = (df_bi_recebimento.join(bi_originais, 
                           (df_bi_recebimento['COD_PARCELAMENTO'] == bi_originais['COD_PARCELAMENTO_ORI']) &
                           (bi_originais['NOM_NATUREZA_TITULO'] == "MENSALIDADE") &
                           (bi_originais['IND_NATUREZA_REMATRICULA'] == "S"), "left"))
    return df_bi_recebimento.drop(bi_originais['COD_PARCELAMENTO_ORI'])

In [20]:
df_bi_recebimento = join_originais(df_bi_recebimento, bi_originais)

In [21]:
#9)	INNER JOIN RAID.RAID.GC_ACCOUNT == bi_contrato GC WITH(NOLOCK) ON GC.ID_CONTRATO = A.CD_ID_CONTRATO
def join_bi_contrato(df_bi_recebimento, bi_contrato):
    df_bi_recebimento = (df_bi_recebimento.join(bi_contrato, 
                           df_bi_recebimento['COD_CONTRATO'] == bi_contrato['COD_CONTRATO'], "left"))
    return df_bi_recebimento.drop(bi_contrato['COD_CONTRATO'])

In [22]:
df_bi_recebimento = join_bi_contrato(df_bi_recebimento, bi_contrato)

In [23]:
#15) Cruzar com a tabela de polos e trazer todas as informaçoes dela
#	select R.*
#			, C.* 
#	FROM df_bi_recebimentos R with(nolock)	
#	left join df_polos_unidades c on c.cod_filial = r.cod_filial
#FAZER WHERE COM ID_COLABORAR, OLIMPO COD
def join_polos_unidades(df_bi_recebimento, polos_unidades):
    df_bi_recebimento = df_bi_recebimento.drop(bi_carteira_cobranca["COD_FILIAL"])
    df_bi_recebimento = (df_bi_recebimento.join(polos_unidades, 
                           (df_bi_recebimento['COD_FILIAL'] == polos_unidades['ID_COLABORAR']) &
                           (df_bi_recebimento['COD_FILIAL'] == polos_unidades['OLIMPO_CODIGO']), "left"))
    
    return df_bi_recebimento.drop(polos_unidades['OLIMPO_CODIGO']).drop(polos_unidades['ID_COLABORAR'])

In [24]:
df_bi_recebimento = join_polos_unidades(df_bi_recebimento, polos_unidades)

In [25]:
#EXECUTE >>

In [26]:
df_bi_recebimento = (df_bi_recebimento
.drop(bi_contrato["NOM_MARCA_CONTRATO"])
.drop(bi_originais["STA_TITULO"])
.drop(bi_originais["COD_TITULO"])
.drop(bi_originais["DAT_VENCIMENTO"])
.drop(bi_originais["NUM_PARCELA"])
.drop(bi_recebimento["NUM_MES_COMPETENCIA"])
.drop(bi_recebimento["NUM_ANO_COMPETENCIA"])
.drop(titulos_vertia["STA_TITULO"])
.drop(titulos_vertia["DAT_PAGAMENTO"])
.drop(bi_contrato["TIP_MATRICULA"])  
.drop(bi_recebimento["COD_TITULO"])
)

In [27]:
#1) df_bi_recebimentos -> filrar somente os titulos ST_TITULO = 'PAGO'
df_bi_recebimento = (df_bi_recebimento.filter(f.col("STA_TITULO") == "PAGO"))    

In [28]:
#2) atualizar o campo DSC_CANAL_REPORT (criar campo)
#	-- recebimento = NOM_GRUPO_USUARIO, NOM_USUARIO_ACORDO
#	-- CASE WHEN  LTRIM(RTRIM(NOM_USUARIO_ACORDO)) = 'raid-portal-aluno' THEN 'raid-portal-aluno'
#	-- 				 WHEN (recebimento ) R.NOM_GRUPO_USUARIO IS NULL AND LTRIM(RTRIM(NOM_USUARIO_ACORDO)) =  'adm' THEN 'adm'
#	-- 			 ELSE 	NOM_USUARIO_ACORDO END AS NOM_USUARIO_ACORDO
df_bi_recebimento = (df_bi_recebimento
                        .withColumn("DSC_CANAL_REPORT",
                        f.when(f.ltrim(f.rtrim(f.col("NOM_USUARIO_ACORDO"))) == "raid-portal-aluno", "raid-portal-aluno")
                        .when((f.col("NOM_GRUPO_USUARIO").isNull()) & (f.ltrim(f.rtrim(f.col("NOM_USUARIO_ACORDO"))) == "adm"), "adm")
                        .otherwise(
                            f.col("NOM_USUARIO_ACORDO")
                        ))) 

In [29]:
#	3) CRIAR campo NOM_CARTEIRA
#	(recebimento)
#			,   CASE WHEN  R.TIP_MATRICULA = 'TIP_MATRICULA' AND R.NOM_MARCA_CONTRATO IN ('ANHANGUERA','UNIDERP')	THEN 'AEDU PRESENCIAL'
#					WHEN  R.TIP_MATRICULA = 'EAD'			AND R.NOM_MARCA_CONTRATO IN ('ANHANGUERA','UNIDERP')	THEN 'AEDU EAD'
#					WHEN  R.TIP_MATRICULA = 'PRESENCIAL'	AND R.NOM_MARCA_CONTRATO IN ('FAMA','PITAGORAS','UNIC','UNIME','UNOPAR','UNIRONDON')	THEN 'DM PRESENCIAL'
#					WHEN  R.TIP_MATRICULA = 'EAD'			AND R.TIP_MATRICULA IN ('UNIME','UNOPAR','PITAGORAS','UNIASSELVI','PITAGORAS','FAMA-MACAPA','UNIC','UNIRONDON') THEN 'UNOPAR EAD'
#				END AS CARTEIRA

df_bi_recebimento = (df_bi_recebimento
                        .withColumn("NOM_CARTEIRA",
                        f.when(("TIP_MATRICULA" == f.lit("TIP_MATRICULA")) & (f.col("NOM_MARCA_CONTRATO").isin('ANHANGUERA','UNIDERP')), "AEDU PRESENCIAL")
                        .when(("TIP_MATRICULA" == f.lit("EAD")) & (f.col("NOM_MARCA_CONTRATO").isin('ANHANGUERA','UNIDERP')), "AEDU EAD")
                        .when(("TIP_MATRICULA" == f.lit("PRESENCIAL")) & (f.col("NOM_MARCA_CONTRATO").isin('FAMA','PITAGORAS','UNIC','UNIME','UNOPAR','UNIRONDON')), "DM PRESENCIAL")
                        .when(("TIP_MATRICULA" == f.lit("EAD")) & (f.col("TIP_MATRICULA").isin('UNIME','UNOPAR','PITAGORAS','UNIASSELVI','PITAGORAS','FAMA-MACAPA','UNIC','UNIRONDON')), "UNOPAR EAD")
                        ))

786742 [Thread-5] WARN  org.apache.spark.sql.Column  - Constructing trivially true equals predicate, 'TIP_MATRICULA = TIP_MATRICULA'. Perhaps you need to use aliases.


In [30]:
#	5) 	
#	UPDATE C
#		SET 
#		 C.NOM_CANAL_COMISSAO = 'RAID_ATP_INTERVALOR'
df_bi_recebimento = (df_bi_recebimento.withColumn("NOM_CANAL_COMISSAO", f.lit("RAID_ATP_INTERVALOR")))

In [31]:
#	5) 	
#	FROM   df_bi_recebimentos	C
#	WHERE  C.NOM_USUARIO_ACORDO LIKE '%-mal%' AND C.DSC_CANAL_REPORT LIKE '%ATP_MALTA%'AND
#			CAST(C.DTA_PAGAMENTO AS DATE) >= CAST('2021-09-10' AS DATE);
    #bi_recebimento = bi_recebimento.filter(
    #f.col("NOM_USUARIO_ACORDO").like("%-mal%")
    #)
df_bi_recebimento = (df_bi_recebimento.filter(
                    (f.col("NOM_USUARIO_ACORDO").like("%-mal%")) &
                    (f.col("DSC_CANAL_REPORT").like("%ATP_MALTA%")) &
                    (f.to_date(f.col('DAT_PAGAMENTO'), 'dd/MM/yyyy').alias('DAT_PAGAMENTO') >= datetime.strptime('2021-09-10', '%Y-%m-%d').date())
                    ))
                      

In [32]:
#8 ) -- MARCA ALUNOS VERTIA
#	UPDATE T
#		SET T.CARTEIRA = 'VERTIA'
df_bi_recebimento = (df_bi_recebimento.withColumn("CARTEIRA", f.lit("VERTIA")))

In [33]:
#9) UPDATE A	
#	CRIAR CAMPO EM RECEBIMENTO: NATUREZA_TITULO = "MENSALIDADE"
#								NATUREZA_REMATRICULA = "S"
df_bi_recebimento = (df_bi_recebimento.withColumn("NATUREZA_TITULO", f.lit("MENSALIDADE")))
df_bi_recebimento = (df_bi_recebimento.withColumn("NATUREZA_REMATRICULA", f.lit("S")))

In [34]:
#13)--DIAS ATRASO
#	-- where T1.DIAS_ATRASO  is null or T1.DIAS_ATRASO  = 0;
df_bi_recebimento = df_bi_recebimento.filter(
    (f.col("NUM_DIAS_ATRASO").isNull())|
    (f.col("NUM_DIAS_ATRASO") == 0)
)

In [44]:
df_bi_recebimento = (df_bi_recebimento.select(
                                "NOM_CARTEIRA",
                                "SIT_PARCELAMENTO",
                                "VL_PRINCIPAL",
                                "VL_DESC_PRINCIPAL",
                                "VL_MULTA",
                                "VL_DESC_MULTA",
                                "VL_JUROS_MORA",
                                "VL_DESC_JUROS_MORA",
                                "VL_JUROS_FINANC",
                                "VL_DESC_JUROS_FINANC",
                                "VL_CORRECAO_MONETARIA",
                                "VL_DESC_CORRECAO_MONETARIA",
                                "TIP_CURSO_ZEUS",
                                "VL_DESC_PRINCIPAL",
                                "VL_DESC_MULTA",
                                "VL_DESC_JUROS_MORA",
                                "VL_DESC_JUROS_FINANC",
                                "VL_DESC_CORRECAO_MONETARIA",
                                "VL_DESC_TAXA_NEGOCIACAO",
                                "VL_MULTA",
                                "VL_JUROS_MORA",
                                "VL_JUROS_FINANC",
                                "VL_CORRECAO_MONETARIA",
                                "VL_TAXA_NEGOCIACAO",
                                "VL_ENTRADA",
                                "VL_PARCELA",
                                "VL_PRINCIPAL",
                                "NUM_DIAS_ATRASO",
                                "cd_ID_CLIENTE ",
                                "cd_ID_CONTRATO",
                                "id_parcelamento",
                                "TIP_MATRICULA",
                                "NOM_PRODUTO_PEP",
                                "COD_TITULO",
                                "NOM_GRUPO_USUARIO",
                                "NOM_USUARIO_ACORDO",
                                "DAT_VENCIMENTO",
                                "NUM_PARCELA",
                                "DAT_PAGAMENTO",
                                "VLR_PAGAMENTO",
                                "NUM_MES_COMPETENCIA",
                                "NUM_ANO_COMPETENCIA",
                                "DSC_NATUREZA_REMATRICULA",
                                "TIP_NATUREZA_TITULO",
                                "DSC_NATUREZA_TITULO",
                                "NUM_MES_COMPETENCIA",
                                "NUM_ANO_COMPETENCIA",
                                "DSC_NATUREZA_REMATRICULA",
                                "TIP_NATUREZA_TITULO",
                                "DSC_NATUREZA_TITULO"    
    ))

In [62]:
df_bi_recebimento.printSchema()

root
 |-- COD_REGISTRO: string (nullable = true)
 |-- TIP_MATRICULA: string (nullable = true)
 |-- NOM_MARCA_CONTRATO: string (nullable = true)
 |-- COD_CLIENTE: string (nullable = true)
 |-- NOM_CLIENTE: string (nullable = true)
 |-- COD_CONTRATO: string (nullable = true)
 |-- NUM_CPF_CNPJ: string (nullable = true)
 |-- DSC_SCORE_GLOBAL: string (nullable = true)
 |-- DSC_SCORE_ENGAJAMENTO: string (nullable = true)
 |-- DSC_CLUSTER: string (nullable = true)
 |-- NOM_PRODUTO_PEP: string (nullable = true)
 |-- DSC_IC_FNU: string (nullable = true)
 |-- DSC_IC_FIES: string (nullable = true)
 |-- COD_TITULO: string (nullable = true)
 |-- NUM_NO_TITULO: string (nullable = true)
 |-- TIP_NATUREZA_TITULO: string (nullable = true)
 |-- DSC_NATUREZA_TITULO: string (nullable = true)
 |-- NUM_MES_COMPETENCIA: string (nullable = true)
 |-- NUM_ANO_COMPETENCIA: string (nullable = true)
 |-- VLR_LIQUIDO: string (nullable = true)
 |-- VLR_SEM_DESC_PONTUAL: string (nullable = true)
 |-- DAT_VENCIMENTO:

In [36]:
df_bi_recebimento.count()

1072069 [Executor task launch worker for task 0.0 in stage 39.0 (TID 423)] ERROR org.apache.spark.executor.Executor  - Exception in task 0.0 in stage 39.0 (TID 423)
java.io.FileNotFoundException: No such file or directory: s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_TITULOS_VERTIA/part-00000-a6e50af3-25b2-46b0-b8cf-e7a60e8be3a1-c000.snappy.parquet
It is possible the underlying files have been updated. You can explicitly invalidate the cache in Spark by running 'REFRESH TABLE tableName' command in SQL or by recreating the Dataset/DataFrame involved.
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.org$apache$spark$sql$execution$datasources$FileScanRDD$$anon$$readCurrentFile(FileScanRDD.scala:124)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:169)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:93)
	at org.apache.spark.sql.execution.FileSourceScanEx

Py4JJavaError: An error occurred while calling o271.count.
: org.apache.spark.sql.catalyst.errors.package$TreeNodeException: execute, tree:
Exchange SinglePartition, ENSURE_REQUIREMENTS, [id=#604]
+- *(14) HashAggregate(keys=[], functions=[partial_count(1)], output=[count#9131L])
   +- *(14) Project
      +- *(14) BroadcastHashJoin [COD_FILIAL#139, COD_FILIAL#139], [ID_COLABORAR#453, OLIMPO_CODIGO#454], LeftOuter, BuildRight, false
         :- *(14) Project [COD_FILIAL#139]
         :  +- *(14) BroadcastHashJoin [COD_CONTRATO#171], [COD_CONTRATO#133], LeftOuter, BuildRight, false
         :     :- *(14) Project [COD_CONTRATO#171]
         :     :  +- *(14) BroadcastHashJoin [COD_PARCELAMENTO#189], [COD_PARCELAMENTO_ORI#311], LeftOuter, BuildRight, false
         :     :     :- *(14) Project [COD_CONTRATO#171, COD_PARCELAMENTO#189]
         :     :     :  +- *(14) BroadcastHashJoin [ltrim(rtrim(COD_CPFCGC_PES#1600, None), None)], [ltrim(rtrim(COD_CPFCGC_PES#299, None), None)], LeftOuter